# Interactive Visualization of Kratos Meshes with PyVista

This Jupyter Notebook demonstrates how to use the newly introduced `KratosMultiphysics.pyvista_utilities` module to seamlessly convert Kratos `ModelPart` objects into standard in-memory PyVista grids for rich 3D visualization, plotting, and post-processing.

### Step 1: Import Modules & Choose Visualization Mode

We import NumPy, PyVista, Kratos Multiphysics, and our PyVista utilities module.

#### 💡 Enabling Interactive 3D Rendering
By default, this notebook renders high-quality **static** images. If you wish to rotate, pan, and zoom the 3D results dynamically directly within this notebook:
1. Install the required interactive packages:
   ```bash
   pip install trame trame-vtk trame-vuetify ipywidgets nest-asyncio
   ```
2. Un-comment and execute the interactive configuration lines in the code cell below:
   ```python
   import nest_asyncio
   nest_asyncio.apply()
   pv.set_jupyter_backend("client")
   ```


In [60]:
import sys
import os

# Resolve the Kratos repo root relative to this notebook's location:
# notebooks/ -> python_scripts/ -> kratos/ -> <repo root>
# NOTE: Use this if self-compiled Kratos is used. If using a system-wide installation, this is not needed.
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
kratos_root = os.path.abspath(os.path.join(notebook_dir, "..", "..", ".."))
kratos_build_path = os.path.join(kratos_root, "bin", "Release")

if kratos_build_path not in sys.path:
    sys.path.insert(0, kratos_build_path)

print(f"Kratos build path: {kratos_build_path}")

Kratos build path: /home/vicente/src/Kratos/bin/Release


In [61]:
import numpy as np
import pyvista as pv
import KratosMultiphysics as KM
import KratosMultiphysics.pyvista_utilities as pv_utils

# Set PyVista theme color
pv.global_theme.color = "white"

# ==========================================================================
# INTERACTIVE 3D VISUALIZATION IN JUPYTER NOTEBOOKS:
# To enable fully interactive 3D controls (rotate, pan, zoom) directly in cells:
# 1. Install required packages in your environment:
#    pip install trame trame-vtk trame-vuetify ipywidgets nest-asyncio
# 2. Un-comment and run the lines below to activate the interactive backend:
#    import nest_asyncio
#    nest_asyncio.apply()
#    pv.set_jupyter_backend("client")
# ==========================================================================

print("Modules imported successfully!")

Modules imported successfully!


### Step 2: Create a Kratos ModelPart with Results Data

We set up a simple 2D triangular mesh with historical solutions step variables: `PRESSURE` (scalar) and `DISPLACEMENT` (vector).

In [62]:
model = KM.Model()
mp = model.CreateModelPart("SimulationResults")

# Add standard solution step variables
mp.AddNodalSolutionStepVariable(KM.DISPLACEMENT)
mp.AddNodalSolutionStepVariable(KM.PRESSURE)

# Create 4 nodes to form a square domain
mp.CreateNewNode(1, 0.0, 0.0, 0.0)
mp.CreateNewNode(2, 2.0, 0.0, 0.0)
mp.CreateNewNode(3, 0.0, 2.0, 0.0)
mp.CreateNewNode(4, 2.0, 2.0, 0.0)

# Assign spatial values for PRESSURE and DISPLACEMENT
for node in mp.Nodes:
    # Pressure increases linearly in both directions
    node.SetSolutionStepValue(KM.PRESSURE, 0, 15.0 * node.X + 25.0 * node.Y)
    # Displacement pulls nodes radially outwards
    node.SetSolutionStepValue(KM.DISPLACEMENT, 0, [0.15 * node.X, 0.15 * node.Y, 0.0])

# Create 2 triangular elements to mesh the domain
properties = mp.GetProperties()[0]
mp.CreateNewElement("Element2D3N", 1, [1, 2, 3], properties)
mp.CreateNewElement("Element2D3N", 2, [2, 4, 3], properties)

print(f"Mesh created with {len(mp.Nodes)} nodes and {len(mp.Elements)} elements.")

Mesh created with 4 nodes and 2 elements.


### Step 3: Convert Kratos ModelPart to PyVista Grid

Using `model_part_to_pyvista`, we convert the `ModelPart` in memory without writing files on disk.

In [63]:
grid = pv_utils.ModelPartToPyVista(
    mp,
    useDeformedConfiguration=False,
    nodalVariables=[KM.PRESSURE, KM.DISPLACEMENT]
)

print("Converted PyVista Unstructured Grid:")
print(grid)

AttributeError: module 'KratosMultiphysics.pyvista_utilities' has no attribute 'ModelPartToPyVista'

### Step 4: Visualizing the Mesh and Results

We can now leverage PyVista's powerful plotting API to render the mesh and overlay results.

In [ ]:
# Plot the PRESSURE scalar field on the undeformed mesh
grid.plot(
    scalars="PRESSURE",
    show_edges=True,
    cmap="viridis",
    cpos="xy",
    text="Kratos Undeformed Mesh - Nodal Pressure"
)

### Step 5: Advanced Post-processing - Warp by Vector

We can warp the mesh directly using the `DISPLACEMENT` vector array to visualize structural deformations.

In [ ]:
# Warp the mesh using the high-level Kratos post-processing helper
warped_mesh = pv_utils.ComputeWarpedMesh(
    mp,
    KM.DISPLACEMENT,
    factor=1.5,
    nodalVariables=[KM.PRESSURE]
)

# Render the warped mesh
warped_mesh.plot(
    scalars="PRESSURE",
    show_edges=True,
    cmap="plasma",
    cpos="xy",
    text="Kratos Warped Mesh - Displaced Shape & Nodal Pressure"
)

### Step 6: Extending to 3D Simulation Models

The `model_part_to_pyvista` converter handles 3D elements and variables seamlessly.
Below we construct a 3D block mesh composed of standard 8-node Hexahedral brick elements (`Element3D8N`) with a simulated temperature field that varies quadratically in space, and visualize internal sections using an orthogonal slice plane.

In [ ]:
# Initialize a new 3D ModelPart
mp_3d = model.CreateModelPart("SimulationResults3D")
mp_3d.AddNodalSolutionStepVariable(KM.TEMPERATURE)
mp_3d.AddNodalSolutionStepVariable(KM.VELOCITY)

# Create a 2x2x2 grid of elements (3x3x3 = 27 nodes)
node_id = 1
for z in range(3):
    for y in range(3):
        for x in range(3):
            n = mp_3d.CreateNewNode(node_id, float(x), float(y), float(z))
            # Temperature field: T = x^2 + y^2 + z^2
            n.SetSolutionStepValue(KM.TEMPERATURE, 0, float(x**2 + y**2 + z**2))
            # Velocity field: steady flow along X with secondary swirl along Y & Z
            n.SetSolutionStepValue(KM.VELOCITY, 0, [2.0, 0.2 * y, -0.1 * x])
            node_id += 1

# Helper to get node ID by x, y, z grid indices
def get_node_id(grid_x, grid_y, grid_z):
    return 1 + grid_x + grid_y * 3 + grid_z * 9

# Create the 8 hexahedral elements
properties = mp_3d.GetProperties()[0]
elem_id = 1
for z in range(2):
    for y in range(2):
        for x in range(2):
            # Vertices ordered according to VTK_HEXAHEDRON standard connectivity
            nodes = [
                get_node_id(x, y, z),
                get_node_id(x + 1, y, z),
                get_node_id(x + 1, y + 1, z),
                get_node_id(x, y + 1, z),
                get_node_id(x, y, z + 1),
                get_node_id(x + 1, y, z + 1),
                get_node_id(x + 1, y + 1, z + 1),
                get_node_id(x, y + 1, z + 1)
            ]
            mp_3d.CreateNewElement("Element3D8N", elem_id, nodes, properties)
            elem_id += 1

print(f"3D Mesh created with {len(mp_3d.Nodes)} nodes and {len(mp_3d.Elements)} elements.")

In [ ]:
# Convert the 3D model part to a PyVista Unstructured Grid
grid_3d = pv_utils.ModelPartToPyVista(
    mp_3d,
    useDeformedConfiguration=False,
    nodalVariables=[KM.TEMPERATURE, KM.VELOCITY]
)

# Extract orthogonal slices using the high-level Kratos post-processing helper
slices = pv_utils.CreateOrthogonalSlices(
    mp_3d,
    x=1.0,
    y=1.0,
    z=1.0,
    nodalVariables=[KM.TEMPERATURE]
)

# Render the 3D orthogonal slices
slices.plot(
    scalars="TEMPERATURE",
    show_edges=True,
    cmap="plasma",
    text="3D Orthogonal Slices - Temperature Nodal Field"
)

#### Isosurface (Contour) Extraction

We can extract constant-value contour surfaces (isosurfaces) across our 3D domain using the high-level **`CreateIsosurfaces`** function. This is particularly useful for visualizing scalar fronts (e.g. isothermal zones).

In [ ]:
# Generate 3D temperature contour surfaces (isosurfaces)
contours = pv_utils.CreateIsosurfaces(
    mp_3d,
    KM.TEMPERATURE,
    valuesOrNumber=4,
    nodalVariables=[KM.TEMPERATURE]
)

# Plot the contour surfaces
contours.plot(
    scalars="TEMPERATURE",
    show_edges=True,
    cmap="plasma",
    text="3D Temperature Isosurfaces"
)

#### Fluid Flow Streamlines

For flow simulations, visualizing vector fields via streamlines is essential. We can trace streamlines using the high-level **`CreateStreamlines`** function, which seeds tracer points in a specified region of a vector field (like `VELOCITY`).

In [ ]:
# Trace lines representing the fluid flow path based on the velocity field
streamlines = pv_utils.CreateStreamlines(
    mp_3d,
    velocityVariable=KM.VELOCITY,
    sourceCenter=[1.0, 1.0, 1.0],
    sourceRadius=0.8,
    nPoints=30
)

# Draw outlines of the domain and overlay the flow streamlines
plotter = pv.Plotter()
plotter.add_mesh(grid_3d.outline(), color="black")
plotter.add_mesh(streamlines, line_width=3.0, cmap="viridis", scalars="VELOCITY")
plotter.add_text("3D Velocity Flow Streamlines", font_size=12)
plotter.show()

### Step 7: Exporting to File

We can easily save the grid to a standard `.vtu` file to share or load in ParaView.

In [ ]:
import tempfile
import os

# Save the warped mesh to a temporary file
temp_dir = tempfile.gettempdir()
output_file = os.path.join(temp_dir, "warped_simulation.vtu")
warped_mesh.save(output_file)

print(f"Mesh successfully saved to: {output_file}")